# Steel Defect Inspection: train, export, benchmark (Colab, free GPU)

**Runtime → Change runtime type → T4 GPU**, then *Run all*. Total ≈ 1 hour (≈ 2 hours with the active-learning experiment).

Produces the numbers for the README and the portfolio:
1. YOLO11 mAP@50 / mAP@50-95 on the held-out **test** split of NEU-DET
2. CPU speed (FPS) of the exported ONNX model, and its accuracy scored with our own evaluator
3. OpenVINO INT8 accuracy and speed (edge)
4. SAM2 pixel masks for detected defects
5. Active learning: mAP vs. number of labelled images, random vs. uncertainty+diversity


In [ ]:
# 1. Setup
!nvidia-smi -L
!git clone -q https://github.com/alihassan891u-lab/steel-defect-inspection.git || true
%cd steel-defect-inspection
!pip -q install -e ".[train,edge,dev]"
import json, os, glob

## 2. Data: NEU-DET
Download the NEU-DET zip (NEU surface defect database, detection version: 1,800 images, 6 classes, VOC boxes; research use) from Kaggle or the original source and either
- upload it in the Files panel as `NEU-DET.zip`, or
- set `KAGGLE_DATASET` below and upload your `kaggle.json`.

The converter searches recursively, so any folder layout works.

In [ ]:
KAGGLE_DATASET = ""   # e.g. "<owner>/<neu-det-dataset>" from the Kaggle page URL; leave empty if you uploaded the zip
if KAGGLE_DATASET:
    !mkdir -p ~/.kaggle && cp kaggle.json ~/.kaggle/ && chmod 600 ~/.kaggle/kaggle.json
    !kaggle datasets download -d {KAGGLE_DATASET} -p data/raw --unzip -q
else:
    !mkdir -p data/raw && unzip -q -o NEU-DET.zip -d data/raw
!python scripts/prepare_data.py --raw data/raw --out data/neu_yolo

## 3. Train YOLO11 and evaluate on the test split

In [ ]:
MODEL, EPOCHS, IMGSZ = "yolo11s.pt", 100, 640
from defectinspect.train import train, val, export
best = train("data/neu_yolo/data.yaml", MODEL, EPOCHS, IMGSZ, name="neu_y11s")
test = val(best, "data/neu_yolo/data.yaml", "test", IMGSZ)
print(json.dumps(test, indent=2))

## 4. Export for CPU / edge and benchmark

In [ ]:
exports = export(best, "data/neu_yolo/data.yaml", IMGSZ)
print(exports)
onnx_path = exports["onnx"]
!python scripts/benchmark_onnx.py --model {onnx_path} --data data/neu_yolo --imgsz {IMGSZ} --threads 2
# smaller input = faster on weak CPUs; NEU images are only 200x200
!python scripts/benchmark_onnx.py --model {onnx_path} --data data/neu_yolo --imgsz {IMGSZ} --threads 2 --runs 200

In [ ]:
# OpenVINO INT8 (Intel CPUs, industrial PCs): accuracy + speed via Ultralytics
from ultralytics import YOLO
if "openvino_int8" in exports:
    r = YOLO(exports["openvino_int8"], task="detect").val(data="data/neu_yolo/data.yaml", split="test", imgsz=IMGSZ, device="cpu")
    int8 = {"map50": round(float(r.box.map50), 4), "map50_95": round(float(r.box.map), 4), "speed_ms": r.speed}
    print(int8)

## 5. SAM2 masks on YOLO detections (defect area, not just boxes)

In [ ]:
import cv2, matplotlib.pyplot as plt
from defectinspect.onnx_detector import OnnxDetector
from defectinspect.segment import SamRefiner
from defectinspect.quality import judge, draw, load_rules
from defectinspect.data import NEU_CLASSES
det, sam, rules = OnnxDetector(onnx_path, NEU_CLASSES, imgsz=IMGSZ), SamRefiner("sam2.1_b.pt"), load_rules("configs/rules.yaml")
files = sorted(glob.glob("data/neu_yolo/images/test/*"))[::40][:8]
fig, axes = plt.subplots(2, 4, figsize=(16, 8))
for ax, f in zip(axes.ravel(), files):
    img = cv2.imread(f)
    insp = judge(sam.refine(img, det.predict(img)), img.shape[:2], rules)
    ax.imshow(cv2.cvtColor(draw(img, insp), cv2.COLOR_BGR2RGB)); ax.set_title(os.path.basename(f)); ax.axis("off")
plt.tight_layout(); os.makedirs("docs", exist_ok=True); plt.savefig("docs/demo_grid.png", dpi=120); plt.show()

## 6. Active learning experiment (optional, ≈ 1 hour)

In [ ]:
RUN_ACTIVE_LEARNING = True
if RUN_ACTIVE_LEARNING:
    from defectinspect.active_learning import run_experiment
    curves = run_experiment("data/neu_yolo", "data/neu_yolo/data.yaml", strategies=("random", "hybrid"),
                            start_frac=0.2, step_frac=0.1, rounds=3, epochs=40, model="yolo11n.pt", imgsz=IMGSZ)
    for s, c in curves.items():
        plt.plot([x["labelled"] for x in c], [x["map50"] for x in c], "o-", label=s)
    plt.xlabel("labelled training images"); plt.ylabel("test mAP@50"); plt.legend(); plt.grid(alpha=.3)
    plt.savefig("docs/active_learning.png", dpi=120); plt.show()

## 7. Collect results (send `results.zip` back)

In [ ]:
summary = {"test": test, "exports": exports}
for f in glob.glob("results/*.json"):
    summary[os.path.basename(f)] = json.load(open(f))
json.dump(summary, open("results/summary.json", "w"), indent=2, default=str)
!cp -r runs/defects/neu_y11s/*.png docs/ 2>/dev/null || true
!zip -qr results.zip results docs
from google.colab import files; files.download("results.zip")